In [1]:
import pandas as pd
import requests
import urllib.parse
import time

# --- CONFIGURATION ---
INPUT_FILE = "New Dataset per Parsa.csv"
OUTPUT_FILE = "SIMA_Master_Semantic_Papers.csv"
EMAIL = "wixloop.contact@gmail.com" 
START_YEAR = 2019 

HEADERS = {'User-Agent': f'mailto:{EMAIL}'}

def get_core_uni_words(uni_string):
    if pd.isna(uni_string): return []
    stopwords = ['università', 'universita', 'degli', 'studi', 'di', 'alma', 'mater', 'studiorum', 'campus', 'libera', 'del', 'della']
    words = str(uni_string).lower().replace("'", " ").replace('"', '').split()
    core_words = [w for w in words if w not in stopwords and len(w) > 3]
    return core_words if core_words else words

def validate_author_soft(results, uni_words):
    if not results: return None
    for result in results[:5]: 
        affiliations = result.get('affiliations') or []
        for aff in affiliations:
            inst = aff.get('institution')
            if inst: 
                inst_name = inst.get('display_name', '').lower()
                if any(w in inst_name for w in uni_words):
                    return result.get('id')
    for result in results[:3]:
        last_inst = result.get('last_known_institution')
        if last_inst and last_inst.get('country_code') == 'IT':
            return result.get('id')
    return results[0].get('id')

def get_author_id_tiered(name, university):
    name_clean = str(name).strip()
    uni_words = get_core_uni_words(university)
    encoded_name = urllib.parse.quote(name_clean)
    
    try:
        url = f"https://api.openalex.org/authors?search={encoded_name}"
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code == 200:
            results = response.json().get('results', [])
            author_id = validate_author_soft(results, uni_words)
            if author_id: return author_id
            
        name_parts = name_clean.split()
        if len(name_parts) > 1:
            flipped = urllib.parse.quote(" ".join(name_parts[1:] + [name_parts[0]]))
            resp_flipped = requests.get(f"https://api.openalex.org/authors?search={flipped}", headers=HEADERS, timeout=10)
            if resp_flipped.status_code == 200:
                results_flipped = resp_flipped.json().get('results', [])
                author_id = validate_author_soft(results_flipped, uni_words)
                if author_id: return author_id
    except Exception as e:
        pass
    return None

def reconstruct_abstract(inverted_index):
    if not inverted_index: return ""
    word_index = [(pos, word) for word, positions in inverted_index.items() for pos in positions]
    word_index.sort()
    return " ".join([w[1] for w in word_index])

def get_author_works(author_id, author_name):
    url = f"https://api.openalex.org/works?filter=author.id:{author_id},from_publication_date:{START_YEAR}-01-01"
    works_data = []
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code == 200:
            for work in response.json().get('results', []):
                primary_topic = work.get('primary_topic', {})
                domain = primary_topic.get('domain', {}).get('display_name', 'Unknown') if primary_topic else 'Unknown'
                field = primary_topic.get('field', {}).get('display_name', 'Unknown') if primary_topic else 'Unknown'
                subfield = primary_topic.get('subfield', {}).get('display_name', 'Unknown') if primary_topic else 'Unknown'
                topic = primary_topic.get('display_name', 'Unknown') if primary_topic else 'Unknown'
                
                works_data.append({
                    'Author_Name': author_name,
                    'OpenAlex_Author_ID': author_id,
                    'Paper_Title': work.get('title', ''),
                    'Publication_Year': work.get('publication_year', ''),
                    'Semantic_Field': field,
                    'Semantic_Subfield': subfield,
                    'Semantic_Topic': topic,
                    'Abstract': reconstruct_abstract(work.get('abstract_inverted_index'))
                })
    except:
        pass
    return works_data

print("Loading full master CSV...")
# THE CHANGE: Removed .head(25) to run all 821 members
df = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')
all_papers = []

print(f"\nInitiating Extraction for {len(df)} researchers. This will take ~10-15 minutes...")
print("-" * 50)

for index, row in df.iterrows():
    name = row.get('Cognome e Nome')
    uni = row.get('Ateneo')
    if pd.isna(name): continue

    author_id = get_author_id_tiered(name, uni)
    if author_id:
        works = get_author_works(author_id, name)
        if works:
            all_papers.extend(works)
            print(f"[{index+1}/{len(df)}] ✅ {name}: {len(works)} papers.")
        else:
            print(f"[{index+1}/{len(df)}] ⚠️ {name}: 0 recent papers.")
    else:
        print(f"[{index+1}/{len(df)}] ❌ {name}: No match found.")
    time.sleep(0.5) 

results_df = pd.DataFrame(all_papers)
results_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"\n✅ RECOVERY COMPLETE! File saved: {OUTPUT_FILE}")

Loading full master CSV...

Initiating Extraction for 821 researchers. This will take ~10-15 minutes...
--------------------------------------------------
[1/821] ❌ ABATECOLA Giampaolo: No match found.
[2/821] ✅ ABBATE Tindara: 25 papers.
[3/821] ✅ ABDULKADER Bisan: 7 papers.
[4/821] ✅ ABRATE Graziano: 16 papers.
[5/821] ✅ ACHARD Paola Olimpia: 6 papers.
[6/821] ✅ ADDIS Michela: 25 papers.
[7/821] ❌ AHMAD POUR Leila: No match found.
[8/821] ✅ AIOLFI Simone: 14 papers.
[9/821] ✅ ALVISI Alberto: 1 papers.
[10/821] ✅ AMATULLI Cesare: 25 papers.
[11/821] ✅ AMENTA Carlo: 9 papers.
[12/821] ✅ AMITRANO Cristina Caterina: 10 papers.
[13/821] ✅ AMORE Mario: 25 papers.
[14/821] ✅ ANCARANI Fabio Guido Ulderico: 11 papers.
[15/821] ✅ ANDREINI Daniela: 24 papers.
[16/821] ✅ ANGELINI Antonella: 7 papers.
[17/821] ✅ ANGRISANI Mariarosalba: 7 papers.
[18/821] ✅ ANNESI Nora: 24 papers.
[19/821] ✅ ANNUNZIATA Eleonora: 25 papers.
[20/821] ✅ ANZIVINO Alessia: 17 papers.
[21/821] ✅ APPOLLONI Andrea: 25 pap

In [2]:
import pandas as pd
import re

# --- 1. CONFIGURATION ---
PAPERS_CSV = "SIMA_Master_Semantic_Papers.csv" 
MASTER_CSV = "New Dataset per Parsa.csv" 
TAXONOMY_CSV = "New taxonomy_SIMA_SIM.csv" 
OUTPUT_FILE = "SIMA_Final_Dataset_821.csv"

print("Step 1: Loading all data...")
master_df = pd.read_csv(MASTER_CSV, encoding='utf-8-sig')
tax_df = pd.read_csv(TAXONOMY_CSV, encoding='utf-8-sig')
papers_df = pd.read_csv(PAPERS_CSV, encoding='utf-8-sig', on_bad_lines='skip')

tax_df.columns = tax_df.columns.str.strip()
master_df.columns = master_df.columns.str.strip()

# Build Taxonomy
taxonomy_paths = []
for index, row in tax_df.iterrows():
    macro = str(row.get('Macro Area', '')).strip()
    meso = str(row.get('Meso Area', '')).strip()
    micro = str(row.get('Micro Topic', '')).strip()
    if macro.lower() == 'nan' or not macro: continue
        
    search_terms = []
    for term in [meso, micro]:
        if term and str(term).lower() != 'nan':
            for part in str(term).split('&'):
                cleaned = part.strip().lower()
                if cleaned: search_terms.append(cleaned)
    taxonomy_paths.append({'Macro': macro, 'Meso': meso, 'Micro': micro, 'Search_Terms': search_terms})

# --- 2. THE MASTER CLASSIFIER ---
print(f"\nStep 2: Processing {len(master_df)} Authors (Strict + Fuzzy Rescue)...")
author_data = []
broken_flags = ['unclassified', 'not specified', 'nan']

for index, row in master_df.iterrows():
    author = row.get('Cognome e Nome')
    if pd.isna(author): continue
    
    assigned_macro = str(row.get('Macro Taxonomy', '')).strip()
    # Lock drops if blank or "Not Specified"
    has_locked_macro = (assigned_macro.lower() not in broken_flags) and (assigned_macro != '')

    author_papers = papers_df[papers_df['Author_Name'] == author]
    
    # Check for 0 papers
    if len(author_papers) == 0:
        author_data.append({
            'Cognome e Nome': author,
            'Final_Macro_Area': "Manual Review - No Papers Found",
            'Final_Meso_Area': "Manual Review - No Papers Found",
            'Final_Micro_Topic': "Manual Review - No Papers Found"
        })
        continue

    # Compile text for this author
    all_strict_text = ""
    openalex_tags = ""
    all_fuzzy_text = ""
    for _, paper in author_papers.iterrows():
        openalex_tags += f" {paper.get('Semantic_Field', '')} {paper.get('Semantic_Subfield', '')} {paper.get('Semantic_Topic', '')}".lower()
        title_abs = f" {paper.get('Paper_Title', '')} {paper.get('Abstract', '')}".lower()
        all_strict_text += f" {openalex_tags} {title_abs}"
        all_fuzzy_text += title_abs

    strict_scores = []
    fuzzy_scores = []

    for path in taxonomy_paths:
        if has_locked_macro and path['Macro'].lower() != assigned_macro.lower():
            continue
            
        strict_score = 0
        fuzzy_score = 0
        
        for term in path['Search_Terms']:
            # PASS 1: Strict Count
            strict_score += len(re.findall(r'\b' + re.escape(term) + r'\b', all_strict_text)) * 10
            
            # PASS 2: Fuzzy/Tag Bonus
            if term in openalex_tags:
                fuzzy_score += 50
            tokens = term.split()
            if all(t in all_fuzzy_text for t in tokens):
                fuzzy_score += 10
            elif any(t in all_fuzzy_text for t in tokens):
                fuzzy_score += 1

        if strict_score > 0: strict_scores.append({'path_dict': path, 'score': strict_score})
        if fuzzy_score > 0: fuzzy_scores.append({'path_dict': path, 'score': fuzzy_score})
    
    # Determine the winner
    if strict_scores:
        strict_scores.sort(key=lambda x: x['score'], reverse=True)
        best = strict_scores[0]['path_dict']
        final_macro = assigned_macro if has_locked_macro else best['Macro']
        final_meso, final_micro = best['Meso'], best['Micro']
    elif fuzzy_scores:
        fuzzy_scores.sort(key=lambda x: x['score'], reverse=True)
        best = fuzzy_scores[0]['path_dict']
        final_macro = assigned_macro if has_locked_macro else best['Macro']
        final_meso, final_micro = best['Meso'], best['Micro']
    else:
        final_macro, final_meso, final_micro = "Manual Review - Off Topic", "Manual Review - Off Topic", "Manual Review - Off Topic"
        
    author_data.append({
        'Cognome e Nome': author,
        'Final_Macro_Area': final_macro,
        'Final_Meso_Area': final_meso,
        'Final_Micro_Topic': final_micro
    })

# --- 3. CSV GENERATION ---
print("\nStep 3: Generating Final Dataset...")
profile_df = pd.DataFrame(author_data)
merged_df = pd.merge(master_df, profile_df, on='Cognome e Nome', how='left')
merged_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print(f"✅ SUCCESS! Full dataset saved to: {OUTPUT_FILE}")

Step 1: Loading all data...

Step 2: Processing 821 Authors (Strict + Fuzzy Rescue)...

Step 3: Generating Final Dataset...
✅ SUCCESS! Full dataset saved to: SIMA_Final_Dataset_821.csv


In [3]:
import pandas as pd

# --- CONFIGURATION ---
INPUT_FILE = "SIMA_Final_Dataset_821.csv" # The output from your Master Classifier
OUTPUT_NO_PAPERS = "Review_No_Papers.csv"
OUTPUT_NOT_SPECIFIED = "Review_Not_Specified.csv"

print("Loading the final dataset...")
df = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')

# --- GROUP 1: Missing / No Papers ---
# Filtering rows where the script explicitly flagged 0 papers
no_papers_df = df[df['Final_Macro_Area'] == 'Manual Review - No Papers Found']

# --- GROUP 2: Not Specified / Off Topic ---
# Keywords that indicate the algorithm failed to classify them
failed_keywords = ['off topic', 'unclassified', 'not specified', 'nan']

def is_not_specified(row):
    macro = str(row.get('Final_Macro_Area', '')).lower()
    
    # Skip the "No Papers" folks since they are already in Group 1
    if 'no papers found' in macro:
        return False 
        
    meso = str(row.get('Final_Meso_Area', '')).lower()
    micro = str(row.get('Final_Micro_Topic', '')).lower()
    
    # Return True if any of the failed keywords appear in their final columns
    return (any(k in macro for k in failed_keywords) or 
            any(k in meso for k in failed_keywords) or 
            any(k in micro for k in failed_keywords))

not_specified_df = df[df.apply(is_not_specified, axis=1)]

# --- EXPORT ---
print(f"Found {len(no_papers_df)} authors with NO PAPERS.")
no_papers_df.to_csv(OUTPUT_NO_PAPERS, index=False, encoding='utf-8-sig')

print(f"Found {len(not_specified_df)} authors labeled NOT SPECIFIED / OFF TOPIC.")
not_specified_df.to_csv(OUTPUT_NOT_SPECIFIED, index=False, encoding='utf-8-sig')

print("\n✅ SUCCESS! Two separate review files have been created.")

Loading the final dataset...
Found 40 authors with NO PAPERS.
Found 208 authors labeled NOT SPECIFIED / OFF TOPIC.

✅ SUCCESS! Two separate review files have been created.


In [5]:
import pandas as pd
import re

# --- 1. CONFIGURATION ---
INPUT_FILE = "SIMA_Taxonomy_Test_25.csv"        
PAPERS_CSV = "SIMA_Master_Semantic_Papers.csv"  
TAXONOMY_CSV = "New taxonomy_SIMA_SIM.csv"      
OUTPUT_FILE = "SIMA_Taxonomy_Cleaned_25.csv"    

print("Step 1: Loading data for the Cleanup Operation...")

df_test = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')
papers_df = pd.read_csv(PAPERS_CSV, encoding='utf-8-sig', on_bad_lines='skip')
tax_df = pd.read_csv(TAXONOMY_CSV, encoding='utf-8-sig')

# Clean columns
tax_df.columns = tax_df.columns.str.strip()

# Build the taxonomy paths with split tokens for fuzzy matching
taxonomy_paths = []
for index, row in tax_df.iterrows():
    macro = str(row.get('Macro Area', '')).strip()
    meso = str(row.get('Meso Area', '')).strip()
    micro = str(row.get('Micro Topic', '')).strip()
    
    if macro.lower() == 'nan' or not macro: continue
        
    search_terms = []
    for term in [meso, micro]:
        if term and str(term).lower() != 'nan':
            for part in str(term).split('&'):
                cleaned = part.strip().lower()
                if cleaned: search_terms.append(cleaned)
                    
    taxonomy_paths.append({
        'Macro': macro,
        'Meso': meso,
        'Micro': micro,
        'Search_Terms': search_terms
    })

# --- 2. EXECUTION (The Rescue Operation) ---
print("\nStep 2: Scanning for broken authors...")

# Words that trigger a rescue and unlock the Macro
broken_flags = ['unclassified', 'not specified', 'nan']

fixed_count = 0
no_paper_count = 0

for index, row in df_test.iterrows():
    author = row.get('Cognome e Nome')
    macro_check = str(row.get('Final_Macro_Area', '')).strip().lower()
    meso_check = str(row.get('Final_Meso_Area', '')).strip().lower()
    micro_check = str(row.get('Final_Micro_Topic', '')).strip().lower()
    
    # Check if ANY of the three levels are broken
    needs_fixing = (any(flag in macro_check for flag in broken_flags) or 
                    any(flag in meso_check for flag in broken_flags) or 
                    any(flag in micro_check for flag in broken_flags))
    
    if needs_fixing:
        assigned_macro = str(row.get('Macro Taxonomy', '')).strip()
        
        # THE FIX: If Alberto wrote "Not Specified", the lock drops.
        has_locked_macro = (assigned_macro.lower() not in broken_flags) and (assigned_macro != '')
        
        # 1. Check for papers
        author_papers = papers_df[papers_df['Author_Name'] == author]
        
        if len(author_papers) == 0:
            df_test.at[index, 'Final_Macro_Area'] = "Manual Review - No Papers Found"
            df_test.at[index, 'Final_Meso_Area'] = "Manual Review - No Papers Found"
            df_test.at[index, 'Final_Micro_Topic'] = "Manual Review - No Papers Found"
            no_paper_count += 1
            continue
            
        # 2. Fuzzy Semantic Search
        path_scores = []
        for path in taxonomy_paths:
            # Respect the Macro lock ONLY IF it's a valid department (e.g. "Marketing")
            if has_locked_macro and path['Macro'].lower() != assigned_macro.lower():
                continue
                
            score = 0
            for _, paper in author_papers.iterrows():
                openalex_tags = f"{paper.get('Semantic_Field', '')} {paper.get('Semantic_Subfield', '')} {paper.get('Semantic_Topic', '')}".lower()
                text = f"{paper.get('Paper_Title', '')} {paper.get('Abstract', '')}".lower()
                
                for term in path['Search_Terms']:
                    if term in openalex_tags:
                        score += 50
                    tokens = term.split()
                    if all(t in text for t in tokens):
                        score += 10
                    elif any(t in text for t in tokens):
                        score += 1
            
            if score > 0:
                path_scores.append({'path_dict': path, 'score': score})
                
        path_scores.sort(key=lambda x: x['score'], reverse=True)
        
        if path_scores:
            best = path_scores[0]['path_dict']
            # Output the algorithm's Macro if the lock was dropped
            df_test.at[index, 'Final_Macro_Area'] = assigned_macro if has_locked_macro else best['Macro']
            df_test.at[index, 'Final_Meso_Area'] = best['Meso']
            df_test.at[index, 'Final_Micro_Topic'] = best['Micro']
            fixed_count += 1
        else:
            df_test.at[index, 'Final_Macro_Area'] = "Manual Review - Off Topic"
            df_test.at[index, 'Final_Meso_Area'] = "Manual Review - Off Topic"
            df_test.at[index, 'Final_Micro_Topic'] = "Manual Review - Off Topic"

# --- 3. CSV GENERATION ---
print(f"\nStep 3: Generating Cleaned File...")
print(f" -> Fixed {fixed_count} authors using Fuzzy Matching.")
print(f" -> Flagged {no_paper_count} authors for Manual Review (0 papers).")

df_test.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print(f"\n✅ SUCCESS! Cleaned file saved to: {OUTPUT_FILE}")

# Preview the rescued rows
print("\n--- PREVIEW OF FIXED ROWS ---")
preview = df_test[['Cognome e Nome', 'Macro Taxonomy', 'Final_Macro_Area', 'Final_Meso_Area', 'Final_Micro_Topic']].head(15)
print(preview.to_string(index=False))

Step 1: Loading data for the Cleanup Operation...

Step 2: Scanning for broken authors...

Step 3: Generating Cleaned File...
 -> Fixed 11 authors using Fuzzy Matching.
 -> Flagged 2 authors for Manual Review (0 papers).

✅ SUCCESS! Cleaned file saved to: SIMA_Taxonomy_Cleaned_25.csv

--- PREVIEW OF FIXED ROWS ---
               Cognome e Nome        Macro Taxonomy                Final_Macro_Area                 Final_Meso_Area               Final_Micro_Topic
          ABATECOLA Giampaolo Management & Strategy Manual Review - No Papers Found Manual Review - No Papers Found Manual Review - No Papers Found
               ABBATE Tindara             Marketing                       Marketing Digital & Data-Driven Marketing          Social Media Marketing
             ABDULKADER Bisan         Not Specified            Finance & Accounting               Corporate Finance    M&A and Corporate Governance
              ABRATE Graziano Management & Strategy           Management & Strategy         